In [43]:
import connectorx as cx
import evadb
import ffnn
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from torch.utils.data import DataLoader
import utils
import load_data_to_db
import collections
import os
import h5py
from abc import ABC, abstractmethod
from models.preprocessing.inputs import SparseFeat, DenseFeat, VarLenSparseFeat
from models.dssm import DSSM_Torch, DSSM_TF, get_var_feature, get_test_var_feature
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf, when
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, FloatType, StringType, IntegerType
from dssm_evadb import DSSM_Moel_Wrapper
import pickle
import multiprocessing as mp
from pipeline import Pipeline
from sklearn.model_selection import train_test_split
import pyarrow.parquet as pq


In [21]:
# Use case 10, trainig query
query_to_fetch_training_data = """
select transaction_id, EXTRACT(HOUR FROM time) / 23 as business_hour_norm, amount / transaction_limit as amount_norm, is_fraud
from tpcxai_financial_account_training join tpcxai_financial_transactions_training on fa_customer_sk=sender_id
"""

In [22]:
df = utils.fetch_data_from_postgres_via_connectorx(query_to_fetch_training_data)

In [24]:
df.head()

,transaction_id,business_hour_norm,amount_norm,is_fraud
0,656162581673,0.391304,0.025804,0
1,656162581673,0.391304,0.038817,0
2,18755723913,0.565217,0.934835,0
3,16112567964,0.608696,0.389995,0
4,509183168001,0.304348,0.000098,1


In [40]:
X_features = df[['business_hour_norm', 'amount_norm']].values
y = df['is_fraud'].values.astype(float)

In [41]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=0)

In [ ]:
model.fit(X_train, y_train, epochs=10, batch_size=512, validation_data=(X_test, y_test))

In [50]:
model.save('../../resources/model/tpcxai_sf1/final/tf/usecase10.h5', include_optimizer=False)
model_weights = model.get_weights()
with h5py.File('../../resources/model/tpcxai_sf1/final/velox/usecase10_ffnn_weight.h5', 'w') as f:
    f.create_dataset('w1', data=model_weights[0])
    f.create_dataset('b1', data=model_weights[1])
    f.create_dataset('w2', data=model_weights[2])
    f.create_dataset('b2', data=model_weights[3])

In [53]:
# Use case 10, serving query
query_to_fetch_serving_data = """
select transaction_id, EXTRACT(HOUR FROM time) / 23 as business_hour_norm, amount / transaction_limit as amount_norm
from tpcxai_financial_account_serving join tpcxai_financial_transactions_serving on fa_customer_sk=sender_id
"""

In [54]:
df_serve = utils.fetch_data_from_postgres_via_connectorx(query_to_fetch_serving_data)

In [56]:
X_serve = df_serve[['business_hour_norm', 'amount_norm']].values
y_pred = model.predict(X_serve)

In [57]:
y_pred

array([[1.2381673e-01],
       [3.7592665e-05],
       [3.6965414e-06],
       ...,
       [5.4672360e-04],
       [2.0319001e-05],
       [1.1504413e-05]], dtype=float32)